# Task 3: ETL Notebook – Profile, Cleanse and Integrate

This section performs the ETL process for the Lotus Care outpatient clinic data. The raw CSV files are loaded and profiled before any changes are made. Data quality problems are then identified and corrected, including repeated bookings, inconsistent dates, inconsistent specialty values, invalid measures and integration issues.

All transformations are performed using Python pandas. The original CSV files are not manually edited.

In [2]:
import pandas as pd
from pathlib import Path

# Full location of the coursework CSV files
DATA_DIR = Path(r"C:\DW\DATA")

appointments_raw = pd.read_csv(DATA_DIR / "appointments_raw.csv")
doctors_raw = pd.read_csv(DATA_DIR / "doctors.csv")
clinics_raw = pd.read_csv(DATA_DIR / "clinics.csv")
patients_raw = pd.read_csv(DATA_DIR / "patients.csv")
patient_updates_raw = pd.read_csv(DATA_DIR / "patient_updates.csv")

print("Raw files loaded successfully.")
print("Appointments:", len(appointments_raw))
print("Doctors:", len(doctors_raw))
print("Clinics:", len(clinics_raw))
print("Patients:", len(patients_raw))
print("Patient updates:", len(patient_updates_raw))

Raw files loaded successfully.
Appointments: 1524
Doctors: 21
Clinics: 5
Patients: 300
Patient updates: 3


## Task 3.1 – Profile the Raw Files

This cell profiles all five raw CSV files before any data cleaning is performed. It displays the structure and data types using `info()`, checks missing values using `isnull().sum()`, checks exact duplicate rows using `duplicated()`, and displays distinct value frequencies for selected categorical columns using `value_counts()`.

Expected output: the structure of each raw file and evidence of any missing values, duplicates or inconsistent categorical values. No data is modified in this step.

In [3]:
raw_files = {
    "appointments_raw.csv": appointments_raw,
    "doctors.csv": doctors_raw,
    "clinics.csv": clinics_raw,
    "patients.csv": patients_raw,
    "patient_updates.csv": patient_updates_raw
}

for file_name, df in raw_files.items():
    print("\n" + "=" * 70)
    print(file_name)
    print("=" * 70)

    print("\nINFO:")
    df.info()

    print("\nMISSING VALUES:")
    print(df.isnull().sum())

    print("\nEXACT DUPLICATE ROWS:")
    print(df.duplicated().sum())


appointments_raw.csv

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        1524 non-null   str    
 1   patient_id            1524 non-null   str    
 2   doctor_id             1524 non-null   str    
 3   clinic_id             1524 non-null   str    
 4   appointment_date      1524 non-null   str    
 5   wait_minutes          1524 non-null   int64  
 6   consultation_minutes  1524 non-null   int64  
 7   fee                   1509 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 135.6 KB

MISSING VALUES:
appointment_id           0
patient_id               0
doctor_id                0
clinic_id                0
appointment_date         0
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64

EXACT DUPLICATE ROWS:
0

doctors.csv

INFO:
<class 'pandas.

### Task 3.1 – Categorical Value Profiling

This cell examines the frequency of important categorical attributes. This helps identify inconsistent spellings, capitalisation and unexpected category values before cleaning.

Expected output: frequency counts for doctor specialties, patient gender, patient district, clinic city and clinic province.

In [4]:
print("DOCTOR SPECIALTIES")
print(doctors_raw["specialty"].value_counts(dropna=False))

print("\nPATIENT GENDER")
print(patients_raw["gender"].value_counts(dropna=False))

print("\nPATIENT DISTRICT")
print(patients_raw["district"].value_counts(dropna=False))

print("\nCLINIC CITY")
print(clinics_raw["city"].value_counts(dropna=False))

print("\nCLINIC PROVINCE")
print(clinics_raw["province"].value_counts(dropna=False))

DOCTOR SPECIALTIES
specialty
General Medicine     4
Paediatrics          3
Cardiology           2
Orthopaedics         2
Neurology            2
Dermatology          2
dermatology          1
CARDIOLOGY           1
 Orthopaedics        1
cardiology           1
Neurology            1
general medicine     1
Name: count, dtype: int64

PATIENT GENDER
gender
F    153
M    147
Name: count, dtype: int64

PATIENT DISTRICT
district
Gampaha       49
Jaffna        40
Kandy         40
Matara        39
Colombo       36
Kalutara      36
Galle         34
Kurunegala    26
Name: count, dtype: int64

CLINIC CITY
city
Colombo       1
Kandy         1
Galle         1
Kurunegala    1
Jaffna        1
Name: count, dtype: int64

CLINIC PROVINCE
province
Western          1
Central          1
Southern         1
North Western    1
Northern         1
Name: count, dtype: int64


### Task 3.1 – Appointment Data Quality Checks

This cell performs additional profiling of the appointment data. It checks repeated bookings at the business grain, date conversion problems, missing or non-numeric measures, and measure values that are zero or negative.

For profiling purposes, a repeated booking is checked using the business grain columns `patient_id`, `doctor_id`, `clinic_id`, and `appointment_date`. No rows are removed in this cell.

Expected output: counts of potential repeated bookings, invalid dates, missing/non-numeric measures and non-positive measure values.

In [5]:
grain_columns = [
    "patient_id",
    "doctor_id",
    "clinic_id",
    "appointment_date"
]

repeated_booking_count = appointments_raw.duplicated(
    subset=grain_columns,
    keep="first"
).sum()

parsed_dates_profile = pd.to_datetime(
    appointments_raw["appointment_date"],
    errors="coerce"
)

invalid_date_count = parsed_dates_profile.isna().sum()

measure_columns = [
    "wait_minutes",
    "consultation_minutes",
    "fee"
]

profile_results = []

profile_results.append({
    "Problem": "Repeated bookings at business grain",
    "Rows affected": int(repeated_booking_count)
})

profile_results.append({
    "Problem": "Invalid appointment dates",
    "Rows affected": int(invalid_date_count)
})

for column in measure_columns:
    numeric_version = pd.to_numeric(
        appointments_raw[column],
        errors="coerce"
    )

    missing_or_non_numeric = numeric_version.isna().sum()
    non_positive = (numeric_version <= 0).sum()

    profile_results.append({
        "Problem": f"{column}: missing or non-numeric",
        "Rows affected": int(missing_or_non_numeric)
    })

    profile_results.append({
        "Problem": f"{column}: zero or negative",
        "Rows affected": int(non_positive)
    })

profile_results.append({
    "Problem": "Doctor specialty text requires standardisation",
    "Rows affected": int(
        (
            doctors_raw["specialty"].astype(str)
            != doctors_raw["specialty"].astype(str).str.strip().str.title()
        ).sum()
    )
})

problem_summary = pd.DataFrame(profile_results)

problem_summary

,Problem,Rows affected
0,Repeated bookings at business grain,24
1,Invalid appointment dates,60
2,wait_minutes: missing or non-numeric,0
3,wait_minutes: zero or negative,49
4,consultation_minutes: missing or non-numeric,0
5,consultation_minutes: zero or negative,0
6,fee: missing or non-numeric,15
7,fee: zero or negative,0
8,Doctor specialty text requires standardisation,6


## Task 3.2 – Remove Repeated Bookings

A working copy of the appointment data is created so that the original raw DataFrame remains unchanged. Repeated bookings are identified using the fact table grain: one patient booking with one doctor at one clinic on one appointment date.

The columns `patient_id`, `doctor_id`, `clinic_id`, and `appointment_date` are therefore used to identify repeated bookings. The first occurrence is retained and subsequent occurrences at the same grain are removed.

Expected output: the appointment row count before cleaning, the row count after cleaning, and the number of repeated bookings removed.

In [6]:
appointments = appointments_raw.copy()
doctors = doctors_raw.copy()
clinics = clinics_raw.copy()
patients = patients_raw.copy()
patient_updates = patient_updates_raw.copy()

grain_columns = [
    "patient_id",
    "doctor_id",
    "clinic_id",
    "appointment_date"
]

before_duplicates = len(appointments)

duplicate_mask = appointments.duplicated(
    subset=grain_columns,
    keep="first"
)

duplicates_removed = duplicate_mask.sum()

appointments = appointments.loc[~duplicate_mask].copy()

after_duplicates = len(appointments)

print("Rows before removing repeated bookings:", before_duplicates)
print("Repeated bookings removed:", duplicates_removed)
print("Rows after removing repeated bookings:", after_duplicates)

Rows before removing repeated bookings: 1524
Repeated bookings removed: 24
Rows after removing repeated bookings: 1500


## Task 3.3 – Standardise Appointment Dates

This cell converts every `appointment_date` value to a pandas date using `pd.to_datetime()`. Invalid values are converted to `NaT` so that conversion failures can be counted. Valid dates are then standardised to the ISO `YYYY-MM-DD` format.

Expected output: zero failed date conversions and appointment dates displayed consistently in `YYYY-MM-DD` format.

In [7]:
# Restore the original date values using appointment_id
# This is necessary because a previous conversion attempt changed
# some unrecognised dates to NaN.

date_lookup = appointments_raw[
    ["appointment_id", "appointment_date"]
].rename(
    columns={"appointment_date": "original_appointment_date"}
)

appointments = appointments.drop(
    columns=["appointment_date"],
    errors="ignore"
)

appointments = appointments.merge(
    date_lookup,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

# Show examples of the original date values
print("Sample original date values:")
print(appointments["original_appointment_date"].head(15))

# Convert mixed date formats
converted_dates = pd.to_datetime(
    appointments["original_appointment_date"],
    format="mixed",
    errors="coerce"
)

failed_dates = converted_dates.isna().sum()

print("\nFailed date conversions:", failed_dates)

if failed_dates == 0:
    print("All appointment dates converted successfully.")
else:
    print("\nDates that still failed:")
    print(
        appointments.loc[
            converted_dates.isna(),
            "original_appointment_date"
        ].value_counts(dropna=False)
    )

# Standardise to YYYY-MM-DD
appointments["appointment_date"] = converted_dates.dt.strftime("%Y-%m-%d")

# Remove temporary original-date column
appointments = appointments.drop(
    columns=["original_appointment_date"]
)

print("\nSample standardised dates:")
print(appointments["appointment_date"].head(15))

Sample original date values:
0      2025-04-25
1      2025-03-10
2      2025-08-19
3      2025-06-09
4      2025-08-20
5      2025-04-29
6     18-Oct-2025
7      2025-10-11
8      2025-03-04
9      2025-06-24
10     2025-08-04
11     2025-10-17
12     2025-10-31
13     2025-11-25
14     2025-07-04
Name: original_appointment_date, dtype: str

Failed date conversions: 0
All appointment dates converted successfully.

Sample standardised dates:
0     2025-04-25
1     2025-03-10
2     2025-08-19
3     2025-06-09
4     2025-08-20
5     2025-04-29
6     2025-10-18
7     2025-10-11
8     2025-03-04
9     2025-06-24
10    2025-08-04
11    2025-10-17
12    2025-10-31
13    2025-11-25
14    2025-07-04
Name: appointment_date, dtype: str


## Task 3.4 – Standardise Doctor Specialty Text

The raw doctor data may contain inconsistent capitalisation or unnecessary spaces in the `specialty` column. This cell first displays the distinct raw specialty values and then standardises them by removing leading/trailing spaces and applying consistent title case.

Expected output: the distinct specialty values before and after cleaning, demonstrating that inconsistent text has been standardised.

In [8]:
specialty_before = sorted(
    doctors["specialty"].dropna().unique()
)

print("Distinct specialties BEFORE standardisation:")
for value in specialty_before:
    print(value)

doctors["specialty"] = (
    doctors["specialty"]
    .str.strip()
    .str.title()
)

specialty_after = sorted(
    doctors["specialty"].dropna().unique()
)

print("\nDistinct specialties AFTER standardisation:")
for value in specialty_after:
    print(value)

Distinct specialties BEFORE standardisation:
 Orthopaedics
CARDIOLOGY
Cardiology
Dermatology
General Medicine
Neurology
Neurology  
Orthopaedics
Paediatrics
cardiology 
dermatology
general medicine 

Distinct specialties AFTER standardisation:
Cardiology
Dermatology
General Medicine
Neurology
Orthopaedics
Paediatrics


## Task 3.5 – Validate the Three Measures

The three fact measures are `wait_minutes`, `consultation_minutes`, and `fee`.

The validation rules used are:

- `wait_minutes` must be numeric and greater than or equal to 0 because a patient may legitimately have no waiting time.
- `consultation_minutes` must be numeric and greater than 0 because a completed consultation should have a positive duration.
- `fee` must be numeric and greater than or equal to 0 because a fee cannot be negative.

The cell converts the measures to numeric values and identifies missing, non-numeric and invalid values before applying any cleaning rule.

Expected output: the number of invalid rows found for each measure and the affected source rows for inspection.

In [9]:
for column in measure_columns:
    appointments[column] = pd.to_numeric(
        appointments[column],
        errors="coerce"
    )

invalid_wait = (
    appointments["wait_minutes"].isna()
    | (appointments["wait_minutes"] < 0)
)

invalid_consultation = (
    appointments["consultation_minutes"].isna()
    | (appointments["consultation_minutes"] <= 0)
)

invalid_fee = (
    appointments["fee"].isna()
    | (appointments["fee"] < 0)
)

print("Invalid wait_minutes rows:", invalid_wait.sum())
print("Invalid consultation_minutes rows:", invalid_consultation.sum())
print("Invalid fee rows:", invalid_fee.sum())

invalid_measure_rows = appointments[
    invalid_wait |
    invalid_consultation |
    invalid_fee
]

print("\nRows containing at least one invalid measure:")
display(invalid_measure_rows)

Invalid wait_minutes rows: 8
Invalid consultation_minutes rows: 0
Invalid fee rows: 15

Rows containing at least one invalid measure:


,appointment_id,patient_id,doctor_id,clinic_id,wait_minutes,consultation_minutes,fee,appointment_date
50,A01332,P0220,D14,C04,43,14,NaN,2025-11-18
51,A01421,P0278,D14,C04,7,33,NaN,2025-12-08
52,A00537,P0223,D11,C01,1,17,NaN,2025-05-14
89,A01148,P0031,D13,C03,1,25,NaN,2025-10-01
179,A00720,P0174,D02,C02,61,14,NaN,2025-06-23
387,A01192,P0223,D01,C03,-9,31,2500.0,2025-10-15
451,A00837,P0217,D12,C02,89,25,NaN,2025-07-18
510,A01446,P0005,D18,C03,-7,31,5000.0,2025-12-17
606,A01024,P0160,D03,C03,4,16,NaN,2025-08-30
657,A01464,P0183,D06,C01,-23,25,5000.0,2025-12-22


### Task 3.5 – Apply Measure Cleaning Rules

Rows containing invalid measure values are excluded from the cleaned appointment dataset because replacing unknown or impossible values with invented values could distort waiting-time, workload and fee analyses.

The cell reports how many rows are affected by each measure rule and the overall appointment row count before and after the measure-cleaning step.

Expected output: the number of rows affected by each rule and the resulting cleaned row count.

In [10]:
measure_changes = pd.DataFrame({
    "Measure": [
        "wait_minutes",
        "consultation_minutes",
        "fee"
    ],
    "Rule": [
        "Must be numeric and >= 0",
        "Must be numeric and > 0",
        "Must be numeric and >= 0"
    ],
    "Rows affected": [
        int(invalid_wait.sum()),
        int(invalid_consultation.sum()),
        int(invalid_fee.sum())
    ]
})

display(measure_changes)

before_measure_cleaning = len(appointments)

invalid_any_measure = (
    invalid_wait |
    invalid_consultation |
    invalid_fee
)

appointments = appointments.loc[
    ~invalid_any_measure
].copy()

after_measure_cleaning = len(appointments)

print("Rows before measure cleaning:", before_measure_cleaning)
print("Rows after measure cleaning:", after_measure_cleaning)
print(
    "Total rows removed:",
    before_measure_cleaning - after_measure_cleaning
)

,Measure,Rule,Rows affected
0,wait_minutes,Must be numeric and >= 0,8
1,consultation_minutes,Must be numeric and > 0,0
2,fee,Must be numeric and >= 0,15


Rows before measure cleaning: 1500
Rows after measure cleaning: 1477
Total rows removed: 23


## Task 3.6 – Integrate Appointments with Doctor Data

This cell joins the cleaned appointment records to `doctors.csv` using `doctor_id`. A left join is initially used so that appointment rows are preserved while checking whether every appointment has a matching doctor.

The row count before and after the join is compared. The merge indicator is also examined to identify appointments that do not have a matching doctor.

Expected output: the row counts before and after the join and the number of matched/unmatched appointment records.

In [11]:
# ---------------------------------------------------------
# Task 3.6 - Join Appointments to Doctors
# ---------------------------------------------------------

# 1. Compare row count before and after the initial join
before_join = len(appointments)

initial_join = appointments.merge(
    doctors,
    on="doctor_id",
    how="left",
    indicator=True
)

after_initial_join = len(initial_join)

print("INITIAL JOIN")
print("Rows before join:", before_join)
print("Rows after join:", after_initial_join)
print("Row count difference:", after_initial_join - before_join)


# 2. Find the cause of any row-count increase
duplicate_doctors = doctors[
    doctors.duplicated(
        subset=["doctor_id"],
        keep=False
    )
].sort_values("doctor_id")

print("\nDUPLICATE DOCTOR RECORDS")
display(duplicate_doctors)


# 3. Fix duplicate doctor IDs
doctors_before_fix = len(doctors)

doctors = doctors.drop_duplicates(
    subset=["doctor_id"],
    keep="first"
).copy()

print("Doctor rows before fix:", doctors_before_fix)
print("Doctor rows after fix:", len(doctors))
print(
    "Duplicate doctor IDs remaining:",
    doctors["doctor_id"].duplicated().sum()
)


# 4. Perform the final validated join
before_final_join = len(appointments)

appointments_doctors = appointments.merge(
    doctors,
    on="doctor_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

after_final_join = len(appointments_doctors)

unmatched_doctors = (
    appointments_doctors["_merge"] == "left_only"
).sum()

print("\nFINAL JOIN")
print("Rows before final join:", before_final_join)
print("Rows after final join:", after_final_join)
print("Row count difference:", after_final_join - before_final_join)
print("Unmatched doctor references:", unmatched_doctors)


# 5. Final verification
if (
    before_final_join == after_final_join
    and unmatched_doctors == 0
):
    print("\nJOIN VERIFICATION PASSED")
    print("Every appointment matched exactly one doctor.")
else:
    print("\nJOIN VERIFICATION FAILED")


# Remove temporary merge-check column
appointments_doctors = appointments_doctors.drop(
    columns=["_merge"]
)

INITIAL JOIN
Rows before join: 1477
Rows after join: 1545
Row count difference: 68

DUPLICATE DOCTOR RECORDS


,doctor_id,doctor_name,specialty
6,D07,Dr. Tharindu Rajapaksha,General Medicine
20,D07,Dr. Tharindu Rajapaksha,General Medicine


Doctor rows before fix: 21
Doctor rows after fix: 20
Duplicate doctor IDs remaining: 0

FINAL JOIN
Rows before final join: 1477
Rows after final join: 1477
Row count difference: 0
Unmatched doctor references: 0

JOIN VERIFICATION PASSED
Every appointment matched exactly one doctor.


# Task 4: Surrogate Keys and Load

## Task 4.1 – Assign Surrogate Keys and Create Warehouse CSV Files

This step prepares the cleaned data for loading into the PostgreSQL star schema.

A surrogate key is assigned to every row in `dim_patient`, `dim_doctor`, `dim_clinic`, and `dim_date`. The source identifiers in the appointment data are then mapped to their corresponding surrogate keys.

The final `fact_appointment` table contains only the four foreign keys (`patient_key`, `doctor_key`, `clinic_key`, and `date_key`) and the three measures (`wait_minutes`, `consultation_minutes`, and `fee`). One CSV file is written for each of the five warehouse tables.

Expected output: five CSV files, dimension and fact row counts, and confirmation that no surrogate-key mappings are missing.

In [12]:
# ---------------------------------------------------------
# Task 4.1 - Assign Surrogate Keys and Export Warehouse CSVs
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd

# Folder for the final warehouse CSV files
OUTPUT_DIR = Path(r"C:\DW\warehouse_csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. CREATE DIM_PATIENT
# =========================================================

dim_patient = patients[
    ["patient_id", "gender", "date_of_birth", "district"]
].drop_duplicates(subset=["patient_id"]).copy()

dim_patient = dim_patient.reset_index(drop=True)

dim_patient.insert(
    0,
    "patient_key",
    range(1, len(dim_patient) + 1)
)


# =========================================================
# 2. CREATE DIM_DOCTOR
# doctors was already cleaned in Task 3.6
# =========================================================

dim_doctor = doctors[
    ["doctor_id", "doctor_name", "specialty"]
].drop_duplicates(subset=["doctor_id"]).copy()

dim_doctor = dim_doctor.reset_index(drop=True)

dim_doctor.insert(
    0,
    "doctor_key",
    range(1, len(dim_doctor) + 1)
)


# =========================================================
# 3. CREATE DIM_CLINIC
# =========================================================

dim_clinic = clinics[
    ["clinic_id", "clinic_name", "city", "province"]
].drop_duplicates(subset=["clinic_id"]).copy()

dim_clinic = dim_clinic.reset_index(drop=True)

dim_clinic.insert(
    0,
    "clinic_key",
    range(1, len(dim_clinic) + 1)
)


# =========================================================
# 4. CREATE DIM_DATE
# One row for every date in 2025
# =========================================================

date_range = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "full_date": date_range
})

# Surrogate key
dim_date.insert(
    0,
    "date_key",
    range(1, len(dim_date) + 1)
)

dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()

# Use YYYY-MM-DD in the CSV
dim_date["full_date"] = (
    dim_date["full_date"]
    .dt.strftime("%Y-%m-%d")
)


# =========================================================
# 5. CREATE FACT_APPOINTMENT
# =========================================================

fact_appointment = appointments.copy()

# Map patient source ID -> patient surrogate key
fact_appointment = fact_appointment.merge(
    dim_patient[["patient_id", "patient_key"]],
    on="patient_id",
    how="left",
    validate="many_to_one"
)

# Map doctor source ID -> doctor surrogate key
fact_appointment = fact_appointment.merge(
    dim_doctor[["doctor_id", "doctor_key"]],
    on="doctor_id",
    how="left",
    validate="many_to_one"
)

# Map clinic source ID -> clinic surrogate key
fact_appointment = fact_appointment.merge(
    dim_clinic[["clinic_id", "clinic_key"]],
    on="clinic_id",
    how="left",
    validate="many_to_one"
)

# Map appointment_date -> date surrogate key
fact_appointment = fact_appointment.merge(
    dim_date[["full_date", "date_key"]],
    left_on="appointment_date",
    right_on="full_date",
    how="left",
    validate="many_to_one"
)


# Keep ONLY the four foreign keys and three measures
fact_appointment = fact_appointment[
    [
        "patient_key",
        "doctor_key",
        "clinic_key",
        "date_key",
        "wait_minutes",
        "consultation_minutes",
        "fee"
    ]
].copy()


# =========================================================
# 6. VERIFY SURROGATE KEY MAPPING
# =========================================================

key_columns = [
    "patient_key",
    "doctor_key",
    "clinic_key",
    "date_key"
]

print("MISSING SURROGATE KEY MAPPINGS")
print(fact_appointment[key_columns].isnull().sum())

if fact_appointment[key_columns].isnull().sum().sum() == 0:
    print("\nSurrogate key mapping PASSED.")
else:
    print("\nWARNING: Missing surrogate key mappings found.")


# =========================================================
# 7. EXPORT ONE CSV PER WAREHOUSE TABLE
# =========================================================

dim_patient.to_csv(
    OUTPUT_DIR / "dim_patient.csv",
    index=False
)

dim_doctor.to_csv(
    OUTPUT_DIR / "dim_doctor.csv",
    index=False
)

dim_clinic.to_csv(
    OUTPUT_DIR / "dim_clinic.csv",
    index=False
)

dim_date.to_csv(
    OUTPUT_DIR / "dim_date.csv",
    index=False
)

fact_appointment.to_csv(
    OUTPUT_DIR / "fact_appointment.csv",
    index=False
)


# =========================================================
# 8. DISPLAY FINAL ROW COUNTS
# =========================================================

print("\nFINAL WAREHOUSE ROW COUNTS")
print("dim_patient:", len(dim_patient))
print("dim_doctor:", len(dim_doctor))
print("dim_clinic:", len(dim_clinic))
print("dim_date:", len(dim_date))
print("fact_appointment:", len(fact_appointment))

print("\nCSV files saved to:")
print(OUTPUT_DIR)

print("\nFiles created:")
for file in OUTPUT_DIR.glob("*.csv"):
    print(file.name)

MISSING SURROGATE KEY MAPPINGS
patient_key    0
doctor_key     0
clinic_key     0
date_key       0
dtype: int64

Surrogate key mapping PASSED.

FINAL WAREHOUSE ROW COUNTS
dim_patient: 300
dim_doctor: 20
dim_clinic: 5
dim_date: 365
fact_appointment: 1477

CSV files saved to:
C:\DW\warehouse_csv

Files created:
dim_clinic.csv
dim_date.csv
dim_doctor.csv
dim_patient.csv
fact_appointment.csv


## Task 7.1 – Patient District Update: Overwrite Approach

This analysis demonstrates the overwrite approach for the three patients in
patient_updates.csv. First, the number of 2025 appointments is counted using
each patient's original district. The updated districts are then applied to a
copy of the patient dimension and the appointment counts are recalculated.

Expected output: a comparison showing the appointment count by district before
and after the overwrite, together with the numerical change for each district.

In [13]:
# Task 7.1 - Compare district appointment totals before and after overwrite

# Use copies so the existing warehouse data is not changed
appointments_t7 = appointments.copy()
patients_before = patients_raw.copy()
updates_t7 = patient_updates_raw.copy()

# ---------------------------------------------------------
# BEFORE overwrite
# ---------------------------------------------------------
before = (
    appointments_t7
    .merge(
        patients_before[["patient_id", "district"]],
        on="patient_id",
        how="left"
    )
    .groupby("district")
    .size()
    .reset_index(name="before_appointments")
)

# ---------------------------------------------------------
# Apply overwrite to a COPY of patients
# ---------------------------------------------------------
patients_after = patients_before.copy()

update_map = dict(
    zip(
        updates_t7["patient_id"],
        updates_t7["new_district"]
    )
)

patients_after["district"] = (
    patients_after["patient_id"]
    .map(update_map)
    .fillna(patients_after["district"])
)

# ---------------------------------------------------------
# AFTER overwrite
# ---------------------------------------------------------
after = (
    appointments_t7
    .merge(
        patients_after[["patient_id", "district"]],
        on="patient_id",
        how="left"
    )
    .groupby("district")
    .size()
    .reset_index(name="after_appointments")
)

# ---------------------------------------------------------
# Compare before and after
# ---------------------------------------------------------
district_comparison = (
    before
    .merge(after, on="district", how="outer")
    .fillna(0)
)

district_comparison["before_appointments"] = (
    district_comparison["before_appointments"].astype(int)
)

district_comparison["after_appointments"] = (
    district_comparison["after_appointments"].astype(int)
)

district_comparison["change"] = (
    district_comparison["after_appointments"]
    - district_comparison["before_appointments"]
)

district_comparison = district_comparison.sort_values("district")

print("District appointment totals before and after overwrite:")
display(district_comparison)

print("\nDistricts whose totals changed:")
display(
    district_comparison[
        district_comparison["change"] != 0
    ]
)

District appointment totals before and after overwrite:


,district,before_appointments,after_appointments,change
0,Colombo,193,198,5
1,Galle,156,151,-5
2,Gampaha,240,240,0
3,Jaffna,207,197,-10
4,Kalutara,175,179,4
5,Kandy,188,188,0
6,Kurunegala,124,124,0
7,Matara,194,200,6



Districts whose totals changed:


,district,before_appointments,after_appointments,change
0,Colombo,193,198,5
1,Galle,156,151,-5
3,Jaffna,207,197,-10
4,Kalutara,175,179,4
7,Matara,194,200,6
